In [1]:
import pandas as pd
import os
import pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA, FastICA
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import IsolationForest

In [2]:
df = pd.read_csv('Encoded_Data.csv')
df.head()

X = df.drop(['HeartDisease'], axis='columns')
Y = df[['HeartDisease']]
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.1)

In [3]:
rf = RandomForestClassifier()
model_rf = rf.fit(X_train, Y_train)
pred_rf = model_rf.predict(X_test)
accuracy_rf = accuracy_score(Y_test, pred_rf)

c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [4]:
pca = PCA(n_components=10)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

rf_pca = RandomForestClassifier()
rf_pca.fit(X_train_pca, Y_train)
pred_pca = rf_pca.predict(X_test_pca)
accuracy_pca = accuracy_score(Y_test, pred_pca)

c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [5]:
ica = FastICA(n_components=10) 
X_train_ica = ica.fit_transform(X_train)
X_test_ica = ica.transform(X_test)

rf_ica = RandomForestClassifier()
rf_ica.fit(X_train_ica, Y_train)
pred_ica = rf_ica.predict(X_test_ica)
accuracy_ica = accuracy_score(Y_test, pred_ica)

c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [6]:
clf = IsolationForest(contamination=0.05, random_state=42)
outliers = clf.fit_predict(df.drop('HeartDisease', axis=1))
df = df[outliers == 1]

imputer = SimpleImputer(strategy='mean')
df_filled = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

corr_matrix = df_filled.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
df_selected = df_filled.drop(to_drop, axis=1)

X = df_selected.drop('HeartDisease', axis=1)
y = df_selected['HeartDisease']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1)

rf_classifier = RandomForestClassifier(n_estimators=100)
rf_classifier.fit(X_train, y_train)

y_pred = rf_classifier.predict(X_test)

accuracy_isolation = accuracy_score(y_test, y_pred)

In [7]:
print("Accuracy:", accuracy_rf)
print("Accuracy with PCA:", accuracy_pca)
print("Accuracy with ICA:", accuracy_ica)
print("Accuracy with Isolation:", accuracy_isolation)
print(max(accuracy_rf, max(accuracy_pca, max(accuracy_ica, accuracy_isolation))))

Accuracy: 0.963076923076923
Accuracy with PCA: 0.9446153846153846
Accuracy with ICA: 0.9538461538461539
Accuracy with Isolation: 0.9741100323624595
0.9741100323624595
